# Investigation 03 — Schema-family confusion analysis, with focus on VERTICALITY

This notebook examines which image-schema families are easiest or hardest and which labels are confused.

Special focus: VERTICALITY, because earlier analysis suggested it is the weakest substantive schema family.

Thesis use:
- Supports the per-schema results section.
- Links empirical performance to the literature on image-schema definitional ambiguity.
- Provides examples and confusion tables for VERTICALITY.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from the project notebooks/ directory.
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = DATA_DIR / "outputs"
PARSED_PATH = OUTPUTS_DIR / "parsed_responses.jsonl"
RAW_PATH = OUTPUTS_DIR / "raw_responses.jsonl"
GOLD_PATH = DATA_DIR / "gold" / "sentences_v1.jsonl"

EXPORT_DIR = OUTPUTS_DIR / "top4_investigations"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON in {path} line {line_no}: {exc}") from exc
    return pd.DataFrame(rows)

def safe_read_jsonl(path: Path) -> pd.DataFrame:
    return read_jsonl(path) if path.exists() else pd.DataFrame()

def prompt_generation(prompt_id) -> str:
    prompt_id = str(prompt_id)
    if "v2" in prompt_id or "abstention" in prompt_id:
        return "v2_abstention"
    if "v1" in prompt_id:
        return "v1"
    return "unknown"

def prompt_base(prompt_id: str) -> str:
    prompt_id = str(prompt_id)
    if "direct_schema" in prompt_id:
        return "direct_schema"
    if "structured_roles" in prompt_id:
        return "structured_roles"
    if "naive" in prompt_id:
        return "naive"
    return "unknown"

def condition_family_from_id(condition_id) -> str:
    condition_id = str(condition_id)
    if "temp_0" in condition_id:
        return "temp_0"
    if "temp_03" in condition_id:
        return "temp_03"
    if "temp_07" in condition_id:
        return "temp_07"
    return condition_id

def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["prompt_generation"] = out["prompt_id"].map(prompt_generation) if "prompt_id" in out.columns else "unknown"
    out["prompt_base"] = out["prompt_id"].map(prompt_base) if "prompt_id" in out.columns else "unknown"
    out["condition_family_short"] = out["condition_id"].map(condition_family_from_id) if "condition_id" in out.columns else "unknown"
    out["is_control"] = out["sentence_type"].eq("control_weak_schema") if "sentence_type" in out.columns else False
    out["is_non_control"] = ~out["is_control"]

    if "schema_present" not in out.columns:
        out["schema_present"] = np.where(out.get("main_image_schema", pd.Series()).eq("NONE"), "no", "yes")
    out["gold_schema_present"] = np.where(out["is_control"], "no", "yes")

    out["schema_present_correct"] = out["schema_present"].eq(out["gold_schema_present"])
    out["primary_schema_correct"] = out["main_image_schema"].eq(out["expected_schema_primary"])
    out["lm_correct"] = out["literal_or_metaphorical"].eq(out["expected_literal_or_metaphorical"])
    out["control_correct"] = out["is_control"] & out["literal_or_metaphorical"].eq("control") & out["main_image_schema"].eq("NONE")
    out["control_false_positive_schema"] = out["is_control"] & out["main_image_schema"].notna() & ~out["main_image_schema"].eq("NONE")
    out["predicted_none"] = out["main_image_schema"].eq("NONE") | out["literal_or_metaphorical"].eq("control") | out["schema_present"].eq("no")
    return out

def pct(x, digits=1):
    if x is None or pd.isna(x):
        return "NA"
    return f"{100*x:.{digits}f}%"

def save_csv(df: pd.DataFrame, filename: str) -> Path:
    path = EXPORT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Wrote: {path}")
    return path

def display_percent_table(df: pd.DataFrame, percent_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in percent_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: pct(x) if x is not None else "NA")
    return out

parsed_all = add_derived_columns(read_jsonl(PARSED_PATH))
structured = parsed_all[parsed_all["parse_status"].eq("parsed")].copy()

print(f"All parsed records: {len(parsed_all)}")
print(f"Structured records: {len(structured)}")

In [ ]:
# Exclude controls for substantive schema-family analysis, but keep NONE separately for abstention discussion.
schema_data = structured.copy()
non_control = schema_data[~schema_data["is_control"]].copy()
print(f"Structured records: {len(schema_data)}")
print(f"Non-control records: {len(non_control)}")

In [ ]:
# Schema family summary by prompt generation.
schema_summary = (
    schema_data.groupby(["expected_schema_primary", "prompt_generation"])
    .agg(
        n=("run_key", "count"),
        primary_schema_accuracy=("primary_schema_correct", "mean"),
        lm_accuracy=("lm_correct", "mean"),
        schema_present_accuracy=("schema_present_correct", "mean"),
        control_false_positive_rate=("control_false_positive_schema", "mean"),
    )
    .reset_index()
    .sort_values(["expected_schema_primary", "prompt_generation"])
)

save_csv(schema_summary, "schema_family_summary_by_generation.csv")
display(display_percent_table(schema_summary, [
    "primary_schema_accuracy", "lm_accuracy", "schema_present_accuracy", "control_false_positive_rate"
]))

In [ ]:
# Confusion matrices by prompt generation.
for generation, g in schema_data.groupby("prompt_generation"):
    print(f"\nConfusion matrix: {generation}")
    matrix = pd.crosstab(
        g["expected_schema_primary"],
        g["main_image_schema"],
        rownames=["gold"],
        colnames=["predicted"],
        dropna=False,
    )
    display(matrix)
    matrix.to_csv(EXPORT_DIR / f"schema_confusion_matrix_{generation}.csv")

In [ ]:
# Confusion matrices by prompt_id.
for prompt_id, g in schema_data.groupby("prompt_id"):
    print(f"\nConfusion matrix: {prompt_id}")
    matrix = pd.crosstab(
        g["expected_schema_primary"],
        g["main_image_schema"],
        rownames=["gold"],
        colnames=["predicted"],
        dropna=False,
    )
    display(matrix)
    matrix.to_csv(EXPORT_DIR / f"schema_confusion_matrix_{prompt_id}.csv")

In [ ]:
# VERTICALITY-specific errors.
verticality = schema_data[schema_data["expected_schema_primary"].eq("VERTICALITY")].copy()
verticality["verticality_correct"] = verticality["main_image_schema"].eq("VERTICALITY")
print(f"VERTICALITY records: {len(verticality)}")
print(f"VERTICALITY accuracy: {verticality['verticality_correct'].mean():.3f}")

verticality_by_prompt = (
    verticality.groupby(["prompt_id", "prompt_generation", "provider"])
    .agg(
        n=("run_key", "count"),
        verticality_accuracy=("verticality_correct", "mean"),
        most_common_prediction=("main_image_schema", lambda s: s.value_counts().index[0] if len(s.value_counts()) else None),
    )
    .reset_index()
    .sort_values(["prompt_id", "provider"])
)

save_csv(verticality_by_prompt, "verticality_accuracy_by_prompt_provider.csv")
display(display_percent_table(verticality_by_prompt, ["verticality_accuracy"]))

In [ ]:
verticality_errors = verticality[~verticality["verticality_correct"]].copy()

verticality_error_labels = (
    verticality_errors.groupby(["prompt_id", "provider", "main_image_schema"])
    .size()
    .reset_index(name="count")
    .sort_values(["prompt_id", "provider", "count"], ascending=[True, True, False])
)
save_csv(verticality_error_labels, "verticality_error_predicted_labels.csv")
display(verticality_error_labels)

In [ ]:
# Join sentence text and export VERTICALITY error examples.
gold = safe_read_jsonl(GOLD_PATH)
verticality_examples = verticality_errors.copy()

text_col = None
if not gold.empty:
    text_col = "text" if "text" in gold.columns else "sentence" if "sentence" in gold.columns else None
    if text_col:
        verticality_examples = verticality_examples.merge(
            gold[["sentence_id", text_col]].rename(columns={text_col: "sentence_text"}),
            on="sentence_id",
            how="left",
        )

example_cols = [
    "run_key", "provider", "model_id", "prompt_id", "condition_id", "sentence_id",
    "sentence_text" if "sentence_text" in verticality_examples.columns else "sentence_id",
    "expected_schema_primary", "main_image_schema", "literal_or_metaphorical",
    "interpretation", "schema_explanation"
]
example_cols = [c for c in example_cols if c in verticality_examples.columns]

save_csv(verticality_examples[example_cols], "verticality_error_examples.csv")
display(verticality_examples[example_cols].head(30))

In [ ]:
# Which VERTICALITY sentences are hardest?
verticality_sentence = (
    verticality.groupby("sentence_id")
    .agg(
        n=("run_key", "count"),
        correct_n=("verticality_correct", "sum"),
        accuracy=("verticality_correct", "mean"),
        predicted_schemas=("main_image_schema", lambda s: "; ".join(sorted(set(map(str, s.dropna()))))),
        providers=("provider", lambda s: "; ".join(sorted(set(map(str, s.dropna()))))),
        prompts=("prompt_id", lambda s: "; ".join(sorted(set(map(str, s.dropna()))))),
    )
    .reset_index()
    .sort_values(["accuracy", "n"], ascending=[True, False])
)

if text_col:
    verticality_sentence = verticality_sentence.merge(
        gold[["sentence_id", text_col]].rename(columns={text_col: "sentence_text"}),
        on="sentence_id",
        how="left",
    )

save_csv(verticality_sentence, "verticality_sentence_difficulty.csv")
display(verticality_sentence.head(30))

In [ ]:
# Plot schema-family primary accuracy.
pivot = schema_summary.pivot_table(
    index="expected_schema_primary",
    columns="prompt_generation",
    values="primary_schema_accuracy"
)
display(pivot)

ax = pivot.plot(kind="bar")
ax.set_title("Primary schema accuracy by gold schema family")
ax.set_xlabel("Gold schema")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Thesis interpretation prompts

Use this notebook to support claims such as:

- The schema inventory is empirically uneven; not all theoretically central schemas are equally recoverable.
- VERTICALITY appears to be a difficult family and should be discussed as a case of scalar/orientational ambiguity.
- If VERTICALITY is often confused with SOURCE_PATH_GOAL, that may indicate a change/path construal.
- If VERTICALITY is confused with SUPPORT_BALANCE, that may indicate evaluative or stability readings.
- If VERTICALITY is confused with FORCE, that may indicate causal intensity or pressure readings.

The most useful thesis examples will be sentence-level VERTICALITY errors with model explanations.